# 13 — Export inference model · Streamlit demo

Turns the 81 MB **training checkpoint** (`02_models/densenet121.keras`, weights + Adam optimizer state) into a ~30 MB **inference-only model** the app can load fast, and produces everything the repo `models/` folder needs.

**Guard rails in this notebook**

1. The split is **loaded** from `00_split/`, never rebuilt. The notebook prints `SPLIT_ID` and **stops** unless it equals `9e33ec57c1ec`.
2. The training checkpoint's MD5 must equal `afe3478450c41d91a6ac1e57a2ada44d` — this is what catches the *other* `densenet121.keras` floating around in Drive.
3. Nothing is written until the re-saved model has produced **identical predictions** to the original on a fixed batch of validation images.
4. The class order and the raw-0–255 input convention are **verified empirically**: the exported model must score ≥ 0.98 accuracy on the verification batch. If the label order or preprocessing assumption were wrong, that number would collapse to ~0.03 — it cannot fail silently.

**Outputs** (written to Drive `deliverable2/06_export/`, then downloaded by hand into the repo):

| File | Goes to |
|---|---|
| `densenet121_inference.keras` | repo `models/` |
| `class_names.json` | repo `models/` |
| `inference_spec.json` | repo `models/` (tells the app: img size, input convention, versions) |
| `MD5SUMS_block.txt` | paste into repo `models/MD5SUMS.txt` |
| `samples/sample_NN_<class>.jpg` ×8 | repo `data/samples/` |

**Before you run:** Runtime ▸ Run all. CPU runtime is fine (~15–25 min, most of it the dataset download). Kaggle Secrets needed as usual.

In [ ]:
# Colab setup: Kaggle credentials from Secrets (with retry) + mount Drive. Harmless when run locally.
import os, time

ON_COLAB = False
try:
    from google.colab import userdata, drive
    ON_COLAB = True
except ModuleNotFoundError:
    print("Not on Colab - using local ~/.kaggle/kaggle.json and a local folder in place of Drive.")

if ON_COLAB:
    ok = False
    for attempt in range(1, 4):
        try:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
            print(f"Kaggle credentials loaded from Colab Secrets (attempt {attempt}).")
            ok = True
            break
        except Exception as e:
            print(f"  attempt {attempt}: Secrets not ready ({type(e).__name__}); retrying in 3s...")
            time.sleep(3)
    if not ok:
        print("Colab Secrets did not respond - re-run this cell (the timeout is almost always transient).")
    drive.mount("/content/drive")

Kaggle credentials loaded from Colab Secrets (attempt 1).
Mounted at /content/drive


In [ ]:
# ---- the one config block: everything below reads from here ----
from pathlib import Path
from collections import deque
import sys, subprocess, hashlib, json, shutil
import numpy as np, pandas as pd

SEED             = 42
SPLIT_ID_EXP     = "9e33ec57c1ec"                        # canonical split fingerprint - hard stop if different
CKPT_MD5_EXP     = "afe3478450c41d91a6ac1e57a2ada44d"    # the ONE true training checkpoint - hard stop if different
N_PARAMS_EXP     = 7_076_454                             # DenseNet-121 head+backbone, 38 classes
IMG_SIZE         = 128
N_VERIFY         = 256                                    # validation images for the equivalence + accuracy checks
ACC_FLOOR        = 0.98                                   # below this, label order / preprocessing is WRONG - stop

# 8 demo samples: substring-matched against the canonical class names (assert exactly one match each)
SAMPLE_CLASS_KEYS = [
    "Tomato___Late_blight", "Tomato___Early_blight",      # the famous confusion pair
    "Apple___Apple_scab", "Grape___Black_rot",
    "Potato___healthy", "Common_rust", "Pepper,_bell___Bacterial_spot", "Leaf_scorch",
]
CONF_FLOOR       = 0.90                                   # a sample must be predicted correctly with >= this confidence

# ---- Drive layout (identical to 00_setup_and_split) ----
_drive     = Path("/content/drive/MyDrive")
DRIVE_ROOT = _drive if _drive.exists() else Path.home()
D2         = DRIVE_ROOT / "plant_recognition" / "deliverable2"
SPLIT_DIR  = D2 / "00_split"
SPLIT_CSV  = {s: SPLIT_DIR / f"split_{s}.csv" for s in ("train", "val", "test")}
CKPT_PATH  = D2 / "02_models" / "densenet121.keras"
EXPORT_DIR = D2 / "06_export"; (EXPORT_DIR / "samples").mkdir(parents=True, exist_ok=True)
OUT_MODEL  = EXPORT_DIR / "densenet121_inference.keras"

# ---- raw images (same local-disk convention as every other notebook) ----
PV_DIR   = Path.home() / "plant_recognition" / "plantvillage"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff"}

np.random.seed(SEED)
print("checkpoint :", CKPT_PATH)
print("export to  :", EXPORT_DIR)

checkpoint : /content/drive/MyDrive/plant_recognition/deliverable2/02_models/densenet121.keras
export to  : /content/drive/MyDrive/plant_recognition/deliverable2/06_export


In [ ]:
# Guard rail 1 - the split fingerprint (loaded, never rebuilt; STOP on mismatch)
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

for s, p in SPLIT_CSV.items():
    assert p.exists(), f"missing {p} - splits must already be frozen on Drive"
digests  = {s: md5(SPLIT_CSV[s]) for s in ("train", "val", "test")}
SPLIT_ID = hashlib.md5("".join(digests[s] for s in ("train", "val", "test")).encode()).hexdigest()[:12]
print("SPLIT_ID =", SPLIT_ID)
assert SPLIT_ID == SPLIT_ID_EXP, f"SPLIT_ID {SPLIT_ID} != canonical {SPLIT_ID_EXP} - WRONG SPLIT, STOP."
print("OK - canonical split confirmed.")

SPLIT_ID = 9e33ec57c1ec
OK - canonical split confirmed.


In [ ]:
# Guard rail 2 - the checkpoint identity (catches the wrong densenet121.keras)
assert CKPT_PATH.exists(), f"checkpoint not found at {CKPT_PATH}"
ckpt_md5 = md5(CKPT_PATH)
print(f"checkpoint md5 = {ckpt_md5}  ({CKPT_PATH.stat().st_size/1e6:.1f} MB)")
assert ckpt_md5 == CKPT_MD5_EXP, (
    f"MD5 mismatch - this is NOT the canonical training checkpoint.\n"
    f"  expected {CKPT_MD5_EXP}\n  got      {ckpt_md5}\n"
    "You are probably pointing at the other densenet121.keras in Drive. STOP.")
print("OK - canonical training checkpoint confirmed.")

checkpoint md5 = afe3478450c41d91a6ac1e57a2ada44d  (85.2 MB)
OK - canonical training checkpoint confirmed.


In [ ]:
# Download PlantVillage (skips if already on disk), locate 'color', canonical class order
if not PV_DIR.exists() or not any(PV_DIR.iterdir()):
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        from kaggle.api.kaggle_api_extended import KaggleApi
    PV_DIR.mkdir(parents=True, exist_ok=True)
    api = KaggleApi(); api.authenticate()
    api.dataset_download_files("abdallahalidev/plantvillage-dataset",
                               path=str(PV_DIR), unzip=True, quiet=False)
    print("Downloaded to", PV_DIR)
else:
    print("Already present:", PV_DIR)

def locate(root, names, max_depth=4):
    q = deque([(root, 0)])
    while q:
        d, depth = q.popleft()
        if d.is_dir() and d.name.lower() in names:
            return d
        if d.is_dir() and depth < max_depth:
            for c in sorted(d.iterdir()):
                if c.is_dir():
                    q.append((c, depth + 1))
    return None

COLOR_DIR   = locate(PV_DIR, {"color"})
assert COLOR_DIR is not None, "Could not find a 'color' folder under PV_DIR."
class_names = sorted([d.name for d in COLOR_DIR.iterdir() if d.is_dir()])   # sorted = canonical label order
name2idx    = {n: i for i, n in enumerate(class_names)}
assert len(class_names) == 38, f"expected 38 classes, found {len(class_names)}"
print("38 classes, first / last:", class_names[0], "/", class_names[-1])

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset


100%|██████████| 2.04G/2.04G [00:15<00:00, 141MB/s]



Downloaded to /root/plant_recognition/plantvillage
38 classes, first / last: Apple___Apple_scab / Tomato___healthy


In [ ]:
# Load checkpoint (compile=False - optimizer state is read but will NOT be re-saved), sanity checks
import tensorflow as tf
print(f"TensorFlow {tf.__version__} | Keras {tf.keras.__version__}")

model = tf.keras.models.load_model(CKPT_PATH, compile=False)
n_params = model.count_params()
print(f"parameters: {n_params:,}")
assert n_params == N_PARAMS_EXP, f"parameter count {n_params:,} != canonical {N_PARAMS_EXP:,} - wrong model. STOP."
assert model.output_shape[-1] == 38, "output layer is not 38-way. STOP."
print("OK - architecture matches the canonical model.")

TensorFlow 2.20.0 | Keras 3.13.2
parameters: 7,076,454
OK - architecture matches the canonical model.


In [ ]:
# Fixed verification batch: first N_VERIFY rows of split_val.csv (deterministic), raw 0-255 float @ 128px
val_df = pd.read_csv(SPLIT_CSV["val"]).head(N_VERIFY)

def load_img(path):
    # split CSVs store the Colab-local absolute path; remap onto this runtime's COLOR_DIR
    rel = Path(path)
    p   = COLOR_DIR / rel.parent.name / rel.name
    img = tf.io.read_file(str(p))
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])          # float, still 0-255: the training convention
    return img

X = tf.stack([load_img(p) for p in val_df["filepath"]])
y = np.array([name2idx[l] for l in val_df["label"]])
print("verification batch:", X.shape)

probs_orig = model.predict(X, batch_size=32, verbose=0)
acc = float((probs_orig.argmax(1) == y).mean())
print(f"accuracy of ORIGINAL checkpoint on batch (raw 0-255 input): {acc:.4f}")
assert acc >= ACC_FLOOR, (
    f"accuracy {acc:.4f} < {ACC_FLOOR} - label order or input convention is WRONG "
    "(preprocessing may not be inside the model graph). Do not export. STOP.")
print("OK - class order and raw-0-255 input convention empirically confirmed.")

verification batch: (256, 128, 128, 3)
accuracy of ORIGINAL checkpoint on batch (raw 0-255 input): 0.9961
OK - class order and raw-0-255 input convention empirically confirmed.


In [ ]:
# Save inference-only, reload, require IDENTICAL predictions before keeping anything
model.save(OUT_MODEL)          # compile=False above -> no optimizer state -> small file
size_mb = OUT_MODEL.stat().st_size / 1e6
print(f"written: {OUT_MODEL.name}  ({size_mb:.1f} MB)")

model2      = tf.keras.models.load_model(OUT_MODEL, compile=False)
probs_export = model2.predict(X, batch_size=32, verbose=0)

if np.array_equal(probs_orig, probs_export):
    print("EQUIVALENCE: bit-identical predictions on the verification batch.")
else:
    diff = float(np.max(np.abs(probs_orig - probs_export)))
    print(f"not bit-identical; max abs diff = {diff:.2e}")
    assert diff < 1e-6, "predictions differ materially - export is NOT equivalent. Deleting."
    print("EQUIVALENCE: numerically identical (diff < 1e-6).")
assert (probs_export.argmax(1) == probs_orig.argmax(1)).all(), "argmax mismatch - STOP."
print(f"exported model accuracy on batch: {float((probs_export.argmax(1)==y).mean()):.4f}")

written: densenet121_inference.keras  (29.8 MB)
EQUIVALENCE: bit-identical predictions on the verification batch.
exported model accuracy on batch: 0.9961


In [ ]:
# class_names.json + inference_spec.json - the contract the Streamlit app codes against
(EXPORT_DIR / "class_names.json").write_text(json.dumps(class_names, indent=2))

spec = {
    "model_file":        OUT_MODEL.name,
    "img_size":          IMG_SIZE,
    "input_convention":  "float32, raw 0-255, RGB, (128,128,3); preprocessing lives INSIDE the model graph",
    "class_order":       "sorted() over the 38 color/ class folder names == class_names.json",
    "n_classes":         38,
    "verified_on":       f"first {N_VERIFY} rows of split_val.csv",
    "verified_accuracy": round(float((probs_export.argmax(1) == y).mean()), 4),
    "split_id":          SPLIT_ID,
    "source_checkpoint_md5": ckpt_md5,
    "tensorflow":        tf.__version__,
    "keras":             tf.keras.__version__,
}
(EXPORT_DIR / "inference_spec.json").write_text(json.dumps(spec, indent=2))
print(json.dumps(spec, indent=2))

{
  "model_file": "densenet121_inference.keras",
  "img_size": 128,
  "input_convention": "float32, raw 0-255, RGB, (128,128,3); preprocessing lives INSIDE the model graph",
  "class_order": "sorted() over the 38 color/ class folder names == class_names.json",
  "n_classes": 38,
  "verified_on": "first 256 rows of split_val.csv",
  "verified_accuracy": 0.9961,
  "split_id": "9e33ec57c1ec",
  "source_checkpoint_md5": "afe3478450c41d91a6ac1e57a2ada44d",
  "tensorflow": "2.20.0",
  "keras": "3.13.2"
}


In [ ]:
# 8 demo samples from the TEST split: correctly predicted with confidence >= CONF_FLOOR (deterministic)
test_df = pd.read_csv(SPLIT_CSV["test"])
chosen  = []
for key in SAMPLE_CLASS_KEYS:
    hits = [c for c in class_names if key.lower() in c.lower()]
    assert len(hits) == 1, f"sample key '{key}' matched {len(hits)} classes: {hits} - fix SAMPLE_CLASS_KEYS"
    cls  = hits[0]
    rows = test_df[test_df["label"] == cls]
    pick = None
    for _, r in rows.iterrows():                              # CSV order -> deterministic
        x = tf.expand_dims(load_img(r["filepath"]), 0)
        p = model2.predict(x, verbose=0)[0]
        if class_names[int(p.argmax())] == cls and float(p.max()) >= CONF_FLOOR:
            pick = (r["filepath"], float(p.max())); break
    assert pick is not None, f"no confident correct test image found for {cls} - lower CONF_FLOOR?"
    chosen.append((cls, *pick))

for i, (cls, path, conf) in enumerate(chosen, 1):
    src = COLOR_DIR / Path(path).parent.name / Path(path).name
    dst = EXPORT_DIR / "samples" / f"sample_{i:02d}_{cls}{src.suffix.lower()}"
    shutil.copy2(src, dst)
    print(f"sample_{i:02d}  conf={conf:.3f}  {cls}")
print("\n8 samples written to", EXPORT_DIR / "samples")

sample_01  conf=1.000  Tomato___Late_blight
sample_02  conf=0.969  Tomato___Early_blight
sample_03  conf=1.000  Apple___Apple_scab
sample_04  conf=1.000  Grape___Black_rot
sample_05  conf=1.000  Potato___healthy
sample_06  conf=1.000  Corn_(maize)___Common_rust_
sample_07  conf=1.000  Pepper,_bell___Bacterial_spot
sample_08  conf=1.000  Strawberry___Leaf_scorch

8 samples written to /content/drive/MyDrive/plant_recognition/deliverable2/06_export/samples


In [ ]:
# MD5s of every export + the block to paste into repo models/MD5SUMS.txt
files = [OUT_MODEL, EXPORT_DIR / "class_names.json", EXPORT_DIR / "inference_spec.json"]
block = [f"{md5(f)}  {f.name}" for f in files]
(EXPORT_DIR / "MD5SUMS_block.txt").write_text("\n".join(block) + "\n")
print("\n".join(block))

print(f'''
DONE. Download from Drive  deliverable2/06_export/  into the repo:
  densenet121_inference.keras  -> models/
  class_names.json             -> models/
  inference_spec.json          -> models/
  MD5SUMS_block.txt            -> paste contents into models/MD5SUMS.txt (replace the TODO lines)
  samples/ (8 images)          -> data/samples/
Also copy 02_models/scratch_cnn.keras -> models/  and add its md5 line:
  (run)  md5sum scratch_cnn.keras
Then pin requirements.txt:  tensorflow-cpu=={tf.__version__}
''')

237803310dc9d75596f8b21762028ea7  densenet121_inference.keras
313b850980f6eb767623555cd53a2138  class_names.json
0d72838f18b867eadec9f877322af8d2  inference_spec.json

DONE. Download from Drive  deliverable2/06_export/  into the repo:
  densenet121_inference.keras  -> models/
  class_names.json             -> models/
  inference_spec.json          -> models/
  MD5SUMS_block.txt            -> paste contents into models/MD5SUMS.txt (replace the TODO lines)
  samples/ (8 images)          -> data/samples/
Also copy 02_models/scratch_cnn.keras -> models/  and add its md5 line:
  (run)  md5sum scratch_cnn.keras
Then pin requirements.txt:  tensorflow-cpu==2.20.0



In [ ]:
print(md5(D2 / "02_models" / "scratch_cnn.keras"), " scratch_cnn.keras")

3f9c5e86266b540e4c0596c8bac09ad2  scratch_cnn.keras
